# Lab 17-06 · Constellations, Error Rates and EVM

Five parts, each built on `modulation.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import modulation as md
rng = np.random.default_rng(1706)

## Part 1 · Constellations

*Predict:* $E_s$, $d_{min}$ and peak-to-average ratio for QPSK, 8-PSK, 16-QAM and 64-QAM as built by the lab.

In [2]:
for name, (pts, lab) in (("QPSK", md.psk(4)), ("8-PSK", md.psk(8)), ("16-QAM", md.qam(16)), ("64-QAM", md.qam(64))):
    print(f"{name:7s}: Es = {md.es(pts):6.2f}, d_min = {md.dmin(pts):.3f}, PAPR = {md.papr_db(pts):.2f} dB")

QPSK   : Es =   1.00, d_min = 1.414, PAPR = 0.00 dB
8-PSK  : Es =   1.00, d_min = 0.765, PAPR = 0.00 dB
16-QAM : Es =  10.00, d_min = 2.000, PAPR = 2.55 dB
64-QAM : Es =  42.00, d_min = 2.000, PAPR = 3.68 dB


## Part 2 · Error rates

*Predict:* the BER curves of BPSK/QPSK, 8-PSK, 16-QAM and 64-QAM against $E_b/N_0$.

In [3]:
grid = np.arange(0, 21, 2.0)
for name, (pts, lab) in (("QPSK", md.psk(4)), ("8-PSK", md.psk(8)), ("16-QAM", md.qam(16)), ("64-QAM", md.qam(64))):
    k = math.log2(len(pts))
    ber = [md.simulate(pts, lab, g + 10 * math.log10(k), 200_000, rng)[1] for g in grid]
    plt.plot(grid, np.maximum(ber, 1e-7), "o-", label=name)
plt.plot(grid, norm.sf(np.sqrt(2 * 10 ** (grid / 10))), "k--", label="Q(sqrt(2Eb/N0))")
plt.yscale("log"); plt.ylim(1e-6, 0.5); plt.xlabel("Eb/N0 (dB)"); plt.ylabel("BER"); plt.legend(); plt.show()
p8, l8 = md.psk(8)
s8, _ = md.simulate(p8, l8, 15.0, 1_000_000, rng)
print(f"8-PSK at Es/N0 = 15 dB: simulated SER {s8:.3e}, Craig {md.ser_psk_craig(8, 10 ** 1.5):.3e}")
assert abs(s8 / md.ser_psk_craig(8, 10 ** 1.5) - 1) < 0.1

8-PSK at Es/N0 = 15 dB: simulated SER 2.443e-03, Craig 2.339e-03


## Part 3 · Gray coding

*Predict:* the BER ratio of natural to Gray labelling for 16-QAM at $E_b/N_0$ = 12 dB.

In [4]:
esdb = 12 + 10 * math.log10(4)
_, bg = md.simulate(*md.qam(16), esdb, 2_000_000, rng)
_, bn = md.simulate(*md.natural_qam(16), esdb, 2_000_000, rng)
print(f"Gray BER {bg:.3e}, natural BER {bn:.3e}, ratio {bn / bg:.2f}")
assert 1.2 < bn / bg < 1.45

Gray BER 1.380e-04, natural BER 1.801e-04, ratio 1.31


## Part 4 · EVM

*Predict:* the EVM of 64-QAM with 2° rms phase noise, with a 1%, 1° I/Q imbalance, and after a soft-clipping amplifier at several back-offs.

In [5]:
pts = md.qam(64)[0]
ref = pts[rng.integers(0, 64, 200_000)]
e_pn = md.evm(md.phase_noise(ref, 2.0, rng), ref)
iq = md.iq_imbalance(ref, 1.01, 1.0)
e_iq_raw = md.evm(iq, ref)
g_iq = np.mean(iq * np.conj(ref)) / np.mean(np.abs(ref) ** 2)   # a receiver normalizes the complex gain first
e_iq = md.evm(iq / g_iq, ref)
print(f"I/Q imbalance before gain/phase normalization: EVM {100 * e_iq_raw:.2f}% (includes the wanted term's gain and phase offset)")
print(f"phase noise 2 deg: EVM {100 * e_pn:.2f}% ({-20 * math.log10(e_pn):.1f} dB)")
print(f"I/Q imbalance 1%, 1 deg: EVM {100 * e_iq:.2f}% ({-20 * math.log10(e_iq):.1f} dB)")
peak = np.max(np.abs(pts))
for backoff_db in (0, 2, 4, 6, 8):
    sat = peak * 10 ** (backoff_db / 20)
    out = md.soft_clip(ref, sat)
    gain = np.mean(out * np.conj(ref)) / np.mean(np.abs(ref) ** 2)   # remove the linear gain before measuring
    print(f"saturation {backoff_db} dB above the peak symbol: EVM {100 * md.evm(out / gain, ref):.2f}%")
assert abs(e_pn - 0.0349) < 0.001 and abs(e_iq - 0.0100) < 0.001

I/Q imbalance before gain/phase normalization: EVM 1.43% (includes the wanted term's gain and phase offset)
phase noise 2 deg: EVM 3.50% (29.1 dB)
I/Q imbalance 1%, 1 deg: EVM 1.00% (40.0 dB)
saturation 0 dB above the peak symbol: EVM 4.97%
saturation 2 dB above the peak symbol: EVM 2.44%
saturation 4 dB above the peak symbol: EVM 1.08%
saturation 6 dB above the peak symbol: EVM 0.45%
saturation 8 dB above the peak symbol: EVM 0.18%


## Part 5 · Noncoherent options

*Predict:* how far DPSK and noncoherent FSK sit from BPSK.

In [6]:
grid = np.arange(4, 13, 2.0)
for g in grid:
    l = 10 ** (g / 10)
    print(f"{g:4.0f} dB: BPSK {norm.sf(math.sqrt(2 * l)):.2e}  DPSK sim {md.simulate_dpsk(g, 1_000_000, rng):.2e} (theory {md.ber_dpsk(l):.2e})  "
          f"noncoherent FSK sim {md.simulate_bfsk(g, 1_000_000, rng, coherent=False):.2e} (theory {md.ber_fsk_noncoherent(l):.2e})")

   4 dB: BPSK 1.25e-02  DPSK sim 4.04e-02 (theory 4.06e-02)  noncoherent FSK sim 1.43e-01 (theory 1.42e-01)
   6 dB: BPSK 2.39e-03  DPSK sim 9.31e-03 (theory 9.33e-03)  noncoherent FSK sim 6.87e-02 (theory 6.83e-02)


   8 dB: BPSK 1.91e-04  DPSK sim 8.76e-04 (theory 9.09e-04)  noncoherent FSK sim 2.13e-02 (theory 2.13e-02)


  10 dB: BPSK 3.87e-06  DPSK sim 2.50e-05 (theory 2.27e-05)  noncoherent FSK sim 3.43e-03 (theory 3.37e-03)


  12 dB: BPSK 9.01e-09  DPSK sim 0.00e+00 (theory 6.54e-08)  noncoherent FSK sim 1.73e-04 (theory 1.81e-04)
